# LeetCode 2609 — Find the Longest Balanced Substring of a Binary String bằng Python

Link đề: https://leetcode.com/problems/find-the-longest-balanced-substring-of-a-binary-string/ · Độ khó: **Easy** · Chủ đề: String

Skeleton LeetCode:

```python
class Solution:
    def findTheLongestBalancedSubstring(self, s: str) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu "xâu con cân bằng": một xâu con liên tiếp mà **mọi chữ số `0` đứng trước mọi chữ số `1`** và **số `0` bằng số `1`**; tức xâu có dạng `0` lặp `k` lần rồi `1` lặp `k` lần.
2. Dùng slicing `s[i:j]`, nhân chuỗi `"0" * k`, toán tử `in` để tìm xâu con, và `count` để đếm ký tự.
3. Nhìn xâu nhị phân dưới dạng các **khối liên tiếp (run)** như `000`, `11`, `0`, `111`; một xâu cân bằng chỉ có thể nằm trên **một khối `0` rồi ngay sau là khối `1`**, độ dài `2 * min(độ dài hai khối)`.
4. Viết 4 cách: vét cạn mọi xâu con, thử `k` từ lớn xuống với `"0"*k + "1"*k`, **một lượt đếm** hai biến `zeros` / `ones`, và nhóm khối bằng `itertools.groupby`.
5. Kiểm chứng bằng cách thử **mọi xâu nhị phân** độ dài nhỏ và test ngẫu nhiên có seed.
6. Tránh các lỗi quen thuộc: đếm toàn bộ `0`/`1` của xâu, không reset bộ đếm khi bắt đầu khối `0` mới, quên nhân đôi (`2 * min`), để `ones` lẫn từ khối trước.

## 2. Đề bài & ví dụ

Cho xâu nhị phân `s` chỉ gồm các ký tự `0` và `1`.

Một **xâu con** (substring: dãy ký tự **liên tiếp**) của `s` được gọi là **cân bằng** nếu **tất cả số `0` đứng trước tất cả số `1`** và **số lượng `0` bằng số lượng `1`** trong xâu con đó. Xâu con **rỗng** cũng được coi là cân bằng.

Trả về **độ dài** của xâu con cân bằng **dài nhất** của `s`.

| Ví dụ | s | Output | Giải thích |
|---|---|---|---|
| 1 | `"01000111"` | `6` | xâu con cân bằng dài nhất là `"000111"` (độ dài 6) |
| 2 | `"00111"` | `4` | xâu con cân bằng dài nhất là `"0011"` (độ dài 4) |
| 3 | `"111"` | `0` | ngoài xâu rỗng không có xâu con cân bằng nào |

Constraints: `1 <= s.length <= 50`; `'0' <= s[i] <= '1'` (tức `s` chỉ chứa `0` và `1`).

In [1]:
s = "01000111"
print("s =", s, "độ dài", len(s))
for sub in ("000111", "0001", "01", "0011", "10"):
    print(f"{sub!r} có trong s? {sub in s}")
print("theo đề, đáp án là 6 (xâu con '000111')")

s = 01000111 độ dài 8
'000111' có trong s? True
'0001' có trong s? True
'01' có trong s? True
'0011' có trong s? True
'10' có trong s? True
theo đề, đáp án là 6 (xâu con '000111')


## 3. Kiến thức nền

### 3.1. Duyệt và cắt chuỗi: `s[i:j]`

`s[i:j]` lấy các ký tự từ chỉ số `i` tới `j - 1`. Mọi xâu con của `s` là `s[i:j]` với `0 <= i < j <= len(s)`. Có `n(n+1)/2` xâu con không rỗng.

In [2]:
s = "0011"
subs = [s[i:j] for i in range(len(s)) for j in range(i + 1, len(s) + 1)]
print(subs, len(subs))

['0', '00', '001', '0011', '0', '01', '011', '1', '11', '1'] 10


### 3.2. Nhân chuỗi `"0" * k` và nối chuỗi `+`

`"0" * 3` cho `"000"`; `"0" * 0` cho chuỗi rỗng. Xâu cân bằng độ dài `2k` chính là `"0" * k + "1" * k`.

In [3]:
for k in range(0, 4):
    print(k, repr("0" * k + "1" * k))

0 ''
1 '01'
2 '0011'
3 '000111'


### 3.3. Tìm xâu con bằng `in` và `str.count`

`t in s` kiểm tra `t` có xuất hiện như xâu con liên tiếp của `s` không (không cần chỉ số). `s.count("0")` đếm số ký tự `0`. Ta cũng có thể dùng `"10" not in t` để kiểm tra "không có `1` đứng trước `0`" (tức mọi `0` trước mọi `1`).

In [4]:
s = "01000111"
print("000111" in s, "0011" in s)
print(s.count("0"), s.count("1"))
print("10" in "0011", "10" in "0101")       # False  True

True True
4 4
False True


### 3.4. Khối liên tiếp (run) và `itertools.groupby`

Xâu `"01000111"` gồm các khối `0 | 1 | 000 | 111`. `itertools.groupby(s)` gom các ký tự **liên tiếp bằng nhau**: mỗi nhóm cho `(ký tự, iterator)`; `len(list(it))` là độ dài khối. Đây gọi là **mã hoá độ dài chạy** (run-length encoding).

In [5]:
from itertools import groupby
s = "01000111"
runs = [(ch, len(list(grp))) for ch, grp in groupby(s)]
print(runs)         # [('0', 1), ('1', 1), ('0', 3), ('1', 3)]

[('0', 1), ('1', 1), ('0', 3), ('1', 3)]


### 3.5. Duyệt một lượt và bộ đếm

Thay vì xét mọi xâu con, ta duyệt từng ký tự một lần và duy trì **hai bộ đếm**: `zeros` (đang có bao nhiêu `0` ở khối `0` hiện tại) và `ones` (bao nhiêu `1` liền sau khối `0` đó). Một xâu cân bằng kết thúc ở ký tự hiện tại có độ dài `2 * min(zeros, ones)`. Khi bắt đầu một khối `0` **mới** (sau khi đã thấy `1`), cả hai bộ đếm phải **reset**.

In [6]:
zeros = ones = 0
for ch in "0011":
    if ch == "0":
        zeros += 1
    else:
        ones += 1
    print(f"ký tự {ch}: zeros={zeros} ones={ones} -> độ dài cân bằng hiện tại = {2 * min(zeros, ones)}")

ký tự 0: zeros=1 ones=0 -> độ dài cân bằng hiện tại = 0
ký tự 0: zeros=2 ones=0 -> độ dài cân bằng hiện tại = 0
ký tự 1: zeros=2 ones=1 -> độ dài cân bằng hiện tại = 2
ký tự 1: zeros=2 ones=2 -> độ dài cân bằng hiện tại = 4


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Xâu cân bằng có dạng `0...01...1` với **số `0` = số `1`**. Nhìn xâu `s` thành các khối:

```
s = 0 1 000 111
      ^^^ ^^^     <- khối 000 rồi khối 111: cân bằng dài nhất là 3 + 3 = 6
    ^   ^         <- khối 0 rồi khối 1:   cân bằng dài nhất là 1 + 1 = 2
```

Một xâu cân bằng phải nằm **trong đúng một cặp khối liên tiếp "`0`-khối rồi `1`-khối"**: nó dùng `k` chữ số `0` cuối của khối `0` và `k` chữ số `1` đầu của khối `1`, với `k <= min(độ dài hai khối)`. Lấy `k` lớn nhất: `k = min(độ dài khối 0, độ dài khối 1)` → độ dài `2 * k`.

Ví dụ 2: `"00111"`: khối `00` rồi khối `111` → `k = min(2, 3) = 2` → `4`.

In [7]:
from itertools import groupby

s = "01000111"
runs = [(ch, len(list(g))) for ch, g in groupby(s)]
print("các khối:", runs)
best = 0
for (c1, n1), (c2, n2) in zip(runs, runs[1:]):
    if c1 == "0" and c2 == "1":
        print(f"khối 0 dài {n1} rồi khối 1 dài {n2} -> cân bằng dài nhất {2 * min(n1, n2)}")
        best = max(best, 2 * min(n1, n2))
print("đáp án:", best)

các khối: [('0', 1), ('1', 1), ('0', 3), ('1', 3)]
khối 0 dài 1 rồi khối 1 dài 1 -> cân bằng dài nhất 2
khối 0 dài 3 rồi khối 1 dài 3 -> cân bằng dài nhất 6
đáp án: 6


## 5. Cách 1 — Brute Force / baseline

Duyệt **mọi xâu con** `s[i:j]` và kiểm tra có cân bằng không: `"10"` **không** xuất hiện (mọi `0` trước mọi `1`) **và** số `0` bằng số `1`. Giữ độ dài lớn nhất.

In [8]:
def is_balanced(t: str) -> bool:
    return "10" not in t and t.count("0") == t.count("1")

def longest_bruteforce(s: str) -> int:
    best = 0                                   # xâu rỗng luôn cân bằng
    n = len(s)
    for i in range(n):
        for j in range(i + 1, n + 1):
            t = s[i:j]
            if is_balanced(t):
                best = max(best, len(t))
    return best

In [9]:
print(longest_bruteforce("01000111"))   # 6
print(longest_bruteforce("00111"))      # 4
print(longest_bruteforce("111"))        # 0

6
4
0


**Độ phức tạp:** có `O(n^2)` xâu con, mỗi xâu cần `O(n)` để cắt và kiểm tra → Time `O(n^3)`; Space `O(n)` cho một xâu con. Với `n <= 50` (khoảng `50^3 / 6 ≈ 20.000` thao tác nhỏ) vẫn tức thì.

**Nút thắt:** ta kiểm tra rất nhiều xâu con **không thể** cân bằng (ví dụ chứa `"10"`). Thay vào đó ta tìm thẳng cấu trúc `0^k 1^k`.

## 6. Ý tưởng thuật toán / bước đột phá

Hai hướng đi thẳng tới đáp án:

1. **Đoán `k`:** xâu cân bằng độ dài `2k` chính là `"0" * k + "1" * k`. Thử `k` từ **lớn xuống nhỏ** (`k <= n // 2`); `k` đầu tiên mà `"0" * k + "1" * k in s` cho đáp án `2k`.
2. **Một lượt, hai bộ đếm:** duyệt từng ký tự. Gặp `0`: nếu trước đó đã có `1` (khối `0` mới bắt đầu) → reset `zeros = ones = 0`, rồi `zeros += 1`. Gặp `1`: `ones += 1` và cập nhật đáp án với `2 * min(zeros, ones)`.

Quy tắc "nếu… thì…": `0` sau `1` → bắt đầu khối mới, reset; `1` → đếm thêm một `1` và thử cập nhật đáp án.

In [10]:
s = "01000111"
for k in range(len(s) // 2, 0, -1):
    target = "0" * k + "1" * k
    print(f"k={k}: tìm {target!r} ->", target in s)
    if target in s:
        print("đáp án:", 2 * k)
        break

k=4: tìm '00001111' -> False
k=3: tìm '000111' -> True
đáp án: 6


## 7. Các cách tiếp theo

Ta có thêm 3 cách: **(2)** thử `k` từ lớn xuống với `"0"*k + "1"*k`, **(3)** một lượt với `zeros`/`ones` (cách nộp), **(4)** nhóm khối bằng `groupby`.

### Cách 2 — Thử `k` từ lớn xuống, tìm `"0"*k + "1"*k` trong `s`

Giải thích từng đoạn:

1. `k` lớn nhất có thể là `len(s) // 2` (cần `2k` ký tự).
2. Với mỗi `k`, dựng xâu mẫu `"0" * k + "1" * k` rồi hỏi `mẫu in s`.
3. `k` đầu tiên (lớn nhất) thoả → trả `2 * k`; nếu không `k` nào → `0`.

In [11]:
def longest_by_pattern(s):
    for k in range(len(s) // 2, 0, -1):
        if "0" * k + "1" * k in s:
            return 2 * k
    return 0

print(longest_by_pattern("01000111"), longest_by_pattern("00111"), longest_by_pattern("111"))

6 4 0


In [12]:
class SolutionPattern:
    def findTheLongestBalancedSubstring(self, s: str) -> int:
        for k in range(len(s) // 2, 0, -1):
            if "0" * k + "1" * k in s:
                return 2 * k
        return 0

Độ phức tạp Cách 2: có `n/2` giá trị `k`, mỗi lần dựng mẫu và tìm trong `s` tốn `O(n)` (có thể coi tìm xâu con trung bình tuyến tính) → Time `O(n^2)`; Space `O(n)`.

### Cách 3 — Một lượt với `zeros` / `ones`

Giải thích từng đoạn:

1. `zeros`, `ones` đếm số `0` ở khối `0` hiện tại và số `1` liền sau nó.
2. Gặp `0`: nếu `ones > 0` thì khối `0` mới → reset cả hai bộ đếm; rồi `zeros += 1`.
3. Gặp `1`: `ones += 1`; xâu cân bằng kết thúc tại đây có độ dài `2 * min(zeros, ones)` → cập nhật `best`.

In [13]:
def longest_counters(s):
    zeros = ones = best = 0
    for ch in s:
        if ch == "0":
            if ones > 0:              # trước đó đã có 1: khối 0 mới bắt đầu
                zeros = ones = 0
            zeros += 1
        else:
            ones += 1
            best = max(best, 2 * min(zeros, ones))
    return best

print(longest_counters("01000111"), longest_counters("00111"), longest_counters("111"))

6 4 0


In [14]:
class SolutionCounters:
    def findTheLongestBalancedSubstring(self, s: str) -> int:
        zeros = ones = best = 0
        for ch in s:
            if ch == "0":
                if ones > 0:
                    zeros = ones = 0
                zeros += 1
            else:
                ones += 1
                best = max(best, 2 * min(zeros, ones))
        return best

Độ phức tạp Cách 3: Time `O(n)` (một lượt); Space `O(1)`.

### Cách 4 — Nhóm khối bằng `itertools.groupby`

Mã hoá độ dài chạy: `runs = [(ký tự, độ dài), ...]`. Mỗi cặp khối liên tiếp có dạng (`0`, `a`) rồi (`1`, `b`) cho xâu cân bằng độ dài `2 * min(a, b)`. Đáp án là `max` các giá trị đó (hoặc `0`).

In [15]:
from itertools import groupby

def longest_runs(s):
    runs = [(ch, len(list(g))) for ch, g in groupby(s)]
    best = 0
    for (c1, n1), (c2, n2) in zip(runs, runs[1:]):
        if c1 == "0" and c2 == "1":
            best = max(best, 2 * min(n1, n2))
    return best

print(longest_runs("01000111"), longest_runs("00111"), longest_runs("111"))

6 4 0


In [16]:
from itertools import groupby

class SolutionRuns:
    def findTheLongestBalancedSubstring(self, s: str) -> int:
        runs = [(ch, len(list(g))) for ch, g in groupby(s)]
        best = 0
        for (c1, n1), (c2, n2) in zip(runs, runs[1:]):
            if c1 == "0" and c2 == "1":
                best = max(best, 2 * min(n1, n2))
        return best

Độ phức tạp Cách 4: Time `O(n)`; Space `O(n)` cho danh sách khối (tối đa `n` khối).

## 8. TRACE từng bước

Trace Cách 3 với `s = "01000111"`.

| ký tự | `zeros` trước | `ones` trước | xử lý | `zeros` sau | `ones` sau | `2*min` | `best` |
|---|---|---|---|---|---|---|---|
| `0` | 0 | 0 | `ones == 0` nên không reset; `zeros += 1` | 1 | 0 | — | 0 |
| `1` | 1 | 0 | `ones += 1` | 1 | 1 | 2 | 2 |
| `0` | 1 | 1 | `ones > 0` → **reset** (0, 0), rồi `zeros += 1` | 1 | 0 | — | 2 |
| `0` | 1 | 0 | `zeros += 1` | 2 | 0 | — | 2 |
| `0` | 2 | 0 | `zeros += 1` | 3 | 0 | — | 2 |
| `1` | 3 | 0 | `ones += 1` | 3 | 1 | 2 | 2 |
| `1` | 3 | 1 | `ones += 1` | 3 | 2 | 4 | 4 |
| `1` | 3 | 2 | `ones += 1` | 3 | 3 | 6 | 6 |

Kết quả `best = 6`.

In [17]:
def trace_counters(s):
    zeros = ones = best = 0
    print(f"s = {s!r}")
    for idx, ch in enumerate(s):
        note = ""
        if ch == "0":
            if ones > 0:
                note = "  <- khối 0 mới: reset"
                zeros = ones = 0
            zeros += 1
        else:
            ones += 1
            best = max(best, 2 * min(zeros, ones))
        print(f"i={idx} ch={ch} zeros={zeros} ones={ones} 2*min={2 * min(zeros, ones)} best={best}{note}")
    return best

trace_counters("01000111")
trace_counters("00111")

s = '01000111'
i=0 ch=0 zeros=1 ones=0 2*min=0 best=0
i=1 ch=1 zeros=1 ones=1 2*min=2 best=2
i=2 ch=0 zeros=1 ones=0 2*min=0 best=2  <- khối 0 mới: reset
i=3 ch=0 zeros=2 ones=0 2*min=0 best=2
i=4 ch=0 zeros=3 ones=0 2*min=0 best=2
i=5 ch=1 zeros=3 ones=1 2*min=2 best=2
i=6 ch=1 zeros=3 ones=2 2*min=4 best=4
i=7 ch=1 zeros=3 ones=3 2*min=6 best=6
s = '00111'
i=0 ch=0 zeros=1 ones=0 2*min=0 best=0
i=1 ch=0 zeros=2 ones=0 2*min=0 best=0
i=2 ch=1 zeros=2 ones=1 2*min=2 best=2
i=3 ch=1 zeros=2 ones=2 2*min=4 best=4
i=4 ch=1 zeros=2 ones=3 2*min=4 best=4


4

In [18]:
trace_counters("111")        # không có 0: zeros luôn 0 nên best = 0
trace_counters("0100011")    # khối 0 mới sau khi đã thấy 1

s = '111'
i=0 ch=1 zeros=0 ones=1 2*min=0 best=0
i=1 ch=1 zeros=0 ones=2 2*min=0 best=0
i=2 ch=1 zeros=0 ones=3 2*min=0 best=0
s = '0100011'
i=0 ch=0 zeros=1 ones=0 2*min=0 best=0
i=1 ch=1 zeros=1 ones=1 2*min=2 best=2
i=2 ch=0 zeros=1 ones=0 2*min=0 best=2  <- khối 0 mới: reset
i=3 ch=0 zeros=2 ones=0 2*min=0 best=2
i=4 ch=0 zeros=3 ones=0 2*min=0 best=2
i=5 ch=1 zeros=3 ones=1 2*min=2 best=2
i=6 ch=1 zeros=3 ones=2 2*min=4 best=4


4

In [19]:
# Trace Cách 4: các khối và cặp (0-khối, 1-khối)
from itertools import groupby
s = "01000111"
runs = [(ch, len(list(g))) for ch, g in groupby(s)]
print("runs =", runs)
for idx in range(len(runs) - 1):
    (c1, n1), (c2, n2) = runs[idx], runs[idx + 1]
    if c1 == "0" and c2 == "1":
        print(f"cặp khối #{idx}: 0 x {n1}, 1 x {n2} -> 2*min = {2 * min(n1, n2)}")
    else:
        print(f"cặp khối #{idx}: {c1} x {n1}, {c2} x {n2} -> không phải (0 rồi 1), bỏ qua")

runs = [('0', 1), ('1', 1), ('0', 3), ('1', 3)]
cặp khối #0: 0 x 1, 1 x 1 -> 2*min = 2
cặp khối #1: 1 x 1, 0 x 3 -> không phải (0 rồi 1), bỏ qua
cặp khối #2: 0 x 3, 1 x 3 -> 2*min = 6


## 9. Code nộp LeetCode

Nộp Cách 3 (một lượt, hai bộ đếm): `O(n)` thời gian, `O(1)` bộ nhớ.

In [20]:
class Solution:
    def findTheLongestBalancedSubstring(self, s: str) -> int:
        zeros = ones = best = 0
        for ch in s:
            if ch == "0":
                if ones > 0:           # khối 0 mới bắt đầu: bỏ khối trước
                    zeros = ones = 0
                zeros += 1
            else:
                ones += 1
                best = max(best, 2 * min(zeros, ones))
        return best

print(Solution().findTheLongestBalancedSubstring("01000111"))   # 6
print(Solution().findTheLongestBalancedSubstring("00111"))      # 4
print(Solution().findTheLongestBalancedSubstring("111"))        # 0

6
4
0


## 10. Vì sao thuật toán đúng

- **Hình dạng xâu cân bằng:** một xâu cân bằng khác rỗng có mọi `0` trước mọi `1` và số `0` = số `1` = `k >= 1`, nên chính là `0^k 1^k` — `k` chữ số `0` liên tiếp rồi `k` chữ số `1` liên tiếp. Trong `s`, `0^k` phải nằm ở **cuối một khối `0`** và `1^k` ở **đầu khối `1` ngay sau đó** (nếu giữa hai phần có ký tự khác, xâu con sẽ không liên tiếp dạng `0^k 1^k`). Vậy `k <= min(độ dài hai khối)`, và `k = min(...)` đạt được. Đây là cơ sở cho Cách 4.
- **Invariant của Cách 3:** sau khi xử lý ký tự thứ `i`: nếu `i` đang ở trong một khối `0` thì `zeros` là số `0` đã thấy trong khối đó và `ones = 0`; nếu `i` đang ở trong một khối `1` thì `zeros` là độ dài khối `0` đứng **ngay trước** khối `1` này (hoặc `0` nếu khối `1` mở đầu xâu) và `ones` là số `1` đã thấy trong khối `1` đó. Khi gặp `0` mà `ones > 0`, khối `1` vừa kết thúc và một khối `0` mới bắt đầu, nên reset cả hai bộ đếm: các xâu cân bằng gắn với khối cũ không còn mở rộng được nữa (chúng đã được ghi nhận vào `best` ở các bước trước).
- **Đúng giá trị cập nhật:** tại mỗi `1`, xâu cân bằng dài nhất **kết thúc tại ký tự này** là `0^k 1^k` với `k = min(zeros, ones)` (dùng `k` chữ số `0` cuối của khối `0` và `k` chữ số `1` cuối) — độ dài `2 * min(zeros, ones)`. Xâu cân bằng nào cũng kết thúc tại một ký tự `1` nào đó (trừ xâu rỗng), nên lấy `max` trên mọi vị trí cho đáp án đúng.
- **Xâu rỗng / không có xâu nào:** `best` khởi tạo `0` (độ dài xâu rỗng). Với `"111"`, `zeros` luôn `0` nên `2 * min(0, ones) = 0`.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(s)`.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn mọi xâu con | `O(n^3)` | `O(n)` |
| 2. Thử `k` giảm dần, `"0"*k + "1"*k in s` | `O(n^2)` | `O(n)` |
| 3. Một lượt, hai bộ đếm | `O(n)` | `O(1)` |
| 4. `groupby` các khối | `O(n)` | `O(n)` |

Với `n <= 50` mọi cách đều tức thì; cách 3 là cách chuẩn khi `n` lớn.

## 12. Bộ test

Test 1: ví dụ đề và các edge case: một ký tự, toàn `0`, toàn `1`, `"10"` (sai thứ tự), `"01"`, nhiều khối, khối `0` dài hơn khối `1` và ngược lại, độ dài tối đa `50`.

In [21]:
impls = [
    ("bruteforce", longest_bruteforce),
    ("pattern", longest_by_pattern),
    ("counters", longest_counters),
    ("runs", longest_runs),
    ("class_pattern", SolutionPattern().findTheLongestBalancedSubstring),
    ("class_counters", SolutionCounters().findTheLongestBalancedSubstring),
    ("class_runs", SolutionRuns().findTheLongestBalancedSubstring),
    ("submit", Solution().findTheLongestBalancedSubstring),
]
tests = [
    ("01000111", 6),
    ("00111", 4),
    ("111", 0),
    ("0", 0),
    ("1", 0),
    ("000", 0),
    ("10", 0),                     # 1 trước 0: không cân bằng
    ("01", 2),
    ("0011", 4),
    ("0101", 2),                   # hai khối "01" nhưng không nối thành 0011
    ("010001", 2),                 # khối 0 mới sau khi đã thấy 1
    ("0011100", 4),                # khối 0 dài 2 rồi 1 dài 3
    ("00011", 4),                  # khối 0 dài hơn khối 1
    ("001110001111", 6),           # hai khối 0-1: (2,3) -> 4 và (3,4) -> 6
    ("0" * 25 + "1" * 25, 50),     # độ dài tối đa, toàn bộ là một xâu cân bằng
    ("1" * 25 + "0" * 25, 0),
]
for name, f in impls:
    for s, expected in tests:
        got = f(s)
        assert got == expected, (name, s, got, expected)
    print(name, "PASS", len(tests), "test")

bruteforce PASS 16 test
pattern PASS 16 test
counters PASS 16 test
runs PASS 16 test
class_pattern PASS 16 test
class_counters PASS 16 test
class_runs PASS 16 test
submit PASS 16 test


Test 2: **vét cạn mọi xâu nhị phân** độ dài `1..12` (tổng `8190` xâu), so với một oracle viết khác hẳn Cách 1: xâu con cân bằng ⇔ ký tự của nó đã **sắp tăng** (`sorted(t) == list(t)`, nghĩa là mọi `0` trước mọi `1`) và số `0` bằng số `1`.

In [22]:
from itertools import product

def oracle(s):
    best = 0
    for i in range(len(s)):
        for j in range(i + 1, len(s) + 1):
            t = s[i:j]
            if list(t) == sorted(t) and t.count("0") == t.count("1"):
                best = max(best, len(t))
    return best

count = 0
for length in range(1, 13):
    for bits in product("01", repeat=length):
        s = "".join(bits)
        expected = oracle(s)
        for name, f in impls:
            assert f(s) == expected, (name, s, f(s), expected)
        count += 1
print("đã so khớp", count, "xâu nhị phân (độ dài 1..12)")

đã so khớp 8190 xâu nhị phân (độ dài 1..12)


Test 3: ngẫu nhiên có seed, **đúng constraints** (`1 <= n <= 50`), sinh theo kiểu "nhiều khối" để có nhiều xâu cân bằng dài; oracle như trên.

In [23]:
import random
random.seed(2609)

def random_blocks(n):
    out = []
    ch = random.choice("01")
    while len("".join(out)) < n:
        out.append(ch * random.randint(1, 8))
        ch = "1" if ch == "0" else "0"
    return "".join(out)[:n]

for _ in range(600):
    n = random.randint(1, 50)
    s = random_blocks(n) if random.random() < 0.7 else "".join(random.choice("01") for _ in range(n))
    expected = oracle(s)
    for name, f in impls:
        assert f(s) == expected, (name, s)
print("600 test ngẫu nhiên: tất cả cách đều khớp oracle")

600 test ngẫu nhiên: tất cả cách đều khớp oracle


Test 4: tính chất — kết quả luôn chẵn và `<= len(s)`; chèn hay xoá một ký tự không làm kết quả đổi quá lớn (ở đây ta kiểm tra: nối thêm `"01"` vào cuối làm kết quả `>= 2`); đảo ngược xâu và đổi `0 <-> 1` cho cùng đáp án (đối xứng).

In [24]:
random.seed(9)
swap = str.maketrans("01", "10")
for _ in range(500):
    n = random.randint(1, 48)
    s = random_blocks(n)
    ans = Solution().findTheLongestBalancedSubstring(s)
    assert ans % 2 == 0 and 0 <= ans <= len(s)
    assert Solution().findTheLongestBalancedSubstring(s + "01") >= 2
    mirrored = s[::-1].translate(swap)                 # đảo ngược + đổi 0 <-> 1
    assert Solution().findTheLongestBalancedSubstring(mirrored) == ans
print("tính chất chẵn / đối xứng gương / thêm '01': PASS")

tính chất chẵn / đối xứng gương / thêm '01': PASS


## 13. Các lỗi người mới hay gặp

### Lỗi 1: đếm **toàn bộ** số `0` và số `1` của xâu

In [25]:
def wrong_global_count(s):
    return 2 * min(s.count("0"), s.count("1"))

print(wrong_global_count("01000111"), "vs đúng", Solution().findTheLongestBalancedSubstring("01000111"))
print(wrong_global_count("10"), "vs đúng", Solution().findTheLongestBalancedSubstring("10"))

8 vs đúng 6
2 vs đúng 0


Vì sao sai: xâu con cân bằng phải **liên tiếp** và **mọi `0` trước mọi `1`**. `"01000111"` có `4` số `0` và `4` số `1` nhưng không có xâu con cân bằng độ dài `8` (`"10"` bị chen giữa). Đếm toàn bộ bỏ qua thứ tự và tính liên tiếp.

### Lỗi 2: không reset bộ đếm khi bắt đầu khối `0` mới

In [26]:
def wrong_no_reset(s):
    zeros = ones = best = 0
    for ch in s:
        if ch == "0":
            zeros += 1                         # SAI: thiếu reset khi ones > 0
        else:
            ones += 1
            best = max(best, 2 * min(zeros, ones))
    return best

print(wrong_no_reset("01000111"), "vs đúng", Solution().findTheLongestBalancedSubstring("01000111"))

8 vs đúng 6


Vì sao sai: bộ đếm cộng dồn qua nhiều khối: `zeros = 4` (`0 + 000`) và `ones = 4` (`1 + 111`) → `8`. Nhưng khối `0` đầu tiên và khối `000` bị ngăn cách bởi `1`, không thể cùng nằm trong một xâu `0^k 1^k`. Khi gặp `0` mà trước đó đã có `1`, phải **reset** cả `zeros` và `ones`.

### Lỗi 3: chỉ reset `zeros`, để `ones` lẫn từ khối trước

In [27]:
def wrong_reset_zeros_only(s):
    zeros = ones = best = 0
    for ch in s:
        if ch == "0":
            if ones > 0:
                zeros = 0                      # SAI: quên reset ones
            zeros += 1
        else:
            ones += 1
            best = max(best, 2 * min(zeros, ones))
    return best

print(wrong_reset_zeros_only("010011"), "vs đúng", Solution().findTheLongestBalancedSubstring("010011"))

2 vs đúng 4


Vì sao sai: `ones` không bao giờ về `0` nên điều kiện `ones > 0` **luôn đúng** sau khối `1` đầu tiên; mỗi ký tự `0` kế tiếp lại reset `zeros` về `0` rồi cộng thành `1`. Với `"010011"`, khối `00` cuối bị đếm thành `zeros = 1` (không phải `2`), nên bỏ lỡ xâu `"0011"` (đáp án đúng `4`, code ra `2`). Phải reset **cả hai** bộ đếm khi bắt đầu khối `0` mới.

### Lỗi 4: quên nhân đôi — trả về `min(zeros, ones)`

In [28]:
def wrong_no_double(s):
    zeros = ones = best = 0
    for ch in s:
        if ch == "0":
            if ones > 0:
                zeros = ones = 0
            zeros += 1
        else:
            ones += 1
            best = max(best, min(zeros, ones))      # SAI: thiếu "2 *"
    return best

print(wrong_no_double("01000111"), "vs đúng", Solution().findTheLongestBalancedSubstring("01000111"))

3 vs đúng 6


Vì sao sai: `min(zeros, ones)` là **số cặp** `0-1` (tức `k`), còn đề hỏi **độ dài** xâu con, bằng `2 * k` (`k` chữ số `0` cộng `k` chữ số `1`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Xâu nhị phân + "liên tiếp, `0` trước `1`, số lượng bằng nhau" → xâu có dạng `0^k 1^k` → nghĩ tới **khối (run)** và `min` của hai khối kề nhau.
- Hai cách làm quen thuộc cho xâu: **thử độ dài / mẫu từ lớn xuống** (đơn giản, `O(n^2)` đủ với `n = 50`), hoặc **một lượt với bộ đếm** (`O(n)`).
- Khi bộ đếm phụ thuộc vào "khối hiện tại", luôn tự hỏi: **khi nào bắt đầu khối mới và cần reset những biến nào?**
- Luôn thử các ví dụ biên: `"10"`, `"0101"`, `"010011"`, `"0"`, `"1"`; chúng bắt rất nhiều lỗi.
- Bài họ hàng: "Contiguous Array" (525, `0`/`1` bằng nhau nhưng thứ tự tuỳ ý), "Count Binary Substrings" (696 — chính là kỹ thuật khối kề nhau), "Max Consecutive Ones" (485).

## 15. Biến thể của bài

1. **Count Binary Substrings** (696): đếm **số lượng** xâu con dạng `0^k 1^k` hoặc `1^k 0^k` — với mỗi cặp khối kề nhau cộng `min(a, b)`.
2. Chấp nhận cả dạng `1^k 0^k` (cả hai thứ tự): mở rộng điều kiện `c1 != c2` thay vì cố định `0` rồi `1`.
3. **Contiguous Array** (525): xâu con có số `0` bằng số `1` **không cần** thứ tự — dùng prefix sum và dict.
4. Trả về **chính xâu con** dài nhất (hoặc vị trí bắt đầu), xem bài tập 5.
5. Xâu cho phép chữ cái khác ngoài `0`/`1` (ví dụ `a^k b^k`): cùng ý tưởng khối kề nhau.
6. Cho **nhiều xâu / luồng ký tự dài**: giữ `zeros`, `ones`, `best` khi đọc lần lượt (Cách 3 vẫn đúng).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn mọi xâu con | kiểm tra từng `s[i:j]` | `O(n^3)` | `O(n)` | hiểu đề, oracle |
| 2. Thử `k` giảm dần | `"0"*k + "1"*k in s` | `O(n^2)` | `O(n)` | viết nhanh, `n` nhỏ |
| 3. Hai bộ đếm một lượt | `zeros`, `ones`, reset khi khối `0` mới | `O(n)` | `O(1)` | cách nộp, tối ưu |
| 4. `groupby` khối | `2 * min` cho cặp khối `0` rồi `1` | `O(n)` | `O(n)` | khi dùng nhiều thông tin về khối |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().findTheLongestBalancedSubstring("00110011")` bằng bao nhiêu?

**Bài 2 (tính tay):** `s = "001110001111"`. Chia thành các khối, tìm các cặp (`0`-khối, `1`-khối) và độ dài cân bằng dài nhất mỗi cặp, rồi kết luận.

**Bài 3 (điền TODO):** hoàn thành `run_lengths(s)` trả về list `[(ký tự, độ dài), ...]` (mã hoá độ dài chạy) **không** dùng `groupby`.

**Bài 4 (tự viết lại):** viết `longest_balanced_again(s)` theo Cách 3 (hai bộ đếm) mà không nhìn lại code.

**Bài 5 (mini challenge):** viết `longest_balanced_text(s)` trả về **chính xâu con cân bằng dài nhất** (nếu có nhiều xâu cùng độ dài, trả về xâu **xuất hiện sớm nhất**; nếu không có, trả về chuỗi rỗng `""`).

In [29]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().findTheLongestBalancedSubstring("00110011"))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def run_lengths(s):
    result = []
    # TODO: duyệt từng ký tự ch của s:
    #   - nếu result rỗng hoặc ch khác ký tự của khối cuối -> thêm (ch, 1)
    #   - ngược lại tăng độ dài của khối cuối thêm 1
    return result

4


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** xâu cân bằng có dạng `0^k 1^k`: một khối `0` rồi ngay một khối `1`. Chỉ cần nhìn cặp khối kề nhau.

**Mức 2 (vừa):** Bài 3: giữ danh sách khối, so `ch` với `result[-1][0]`; vì tuple bất biến nên thay phần tử cuối bằng `(ch, độ dài + 1)`. Bài 5: duyệt khối với vị trí bắt đầu; với cặp (`0`, `a`) rồi (`1`, `b`) bắt đầu tại `p`: xâu cân bằng dài nhất là `s[p + a - k : p + a + k]` với `k = min(a, b)`.

**Mức 3 (gần đáp án):** Bài 1: khối `00 | 11 | 00 | 11` → hai cặp đều `2 * 2 = 4` ⇒ `4`. Bài 2: khối `00 | 111 | 000 | 1111`: cặp 1: `2 * min(2, 3) = 4`; cặp 2 (`000` rồi `1111`): `2 * min(3, 4) = 6` ⇒ `6`.

## 19. Đáp án tham khảo cho bài tập

In [30]:
# Bài 1
assert Solution().findTheLongestBalancedSubstring("00110011") == 4

# Bài 2: 00 | 111 | 000 | 1111 -> max(4, 6) = 6
assert Solution().findTheLongestBalancedSubstring("001110001111") == 6

# Bài 3
def run_lengths_answer(s):
    result = []
    for ch in s:
        if not result or result[-1][0] != ch:
            result.append((ch, 1))
        else:
            result[-1] = (ch, result[-1][1] + 1)
    return result

assert run_lengths_answer("01000111") == [("0", 1), ("1", 1), ("0", 3), ("1", 3)]
assert run_lengths_answer("1") == [("1", 1)]
assert run_lengths_answer("0011") == [("0", 2), ("1", 2)]

# Bài 4
def longest_balanced_again(s):
    zeros = ones = best = 0
    for ch in s:
        if ch == "0":
            if ones > 0:
                zeros = ones = 0
            zeros += 1
        else:
            ones += 1
            best = max(best, 2 * min(zeros, ones))
    return best

for length in range(1, 11):
    for bits in product("01", repeat=length):
        t = "".join(bits)
        assert longest_balanced_again(t) == Solution().findTheLongestBalancedSubstring(t)

# Bài 5
def longest_balanced_text(s):
    runs = run_lengths_answer(s)
    best_len, best_text = 0, ""
    pos = 0
    for idx, (ch, length) in enumerate(runs):
        if ch == "0" and idx + 1 < len(runs) and runs[idx + 1][0] == "1":
            k = min(length, runs[idx + 1][1])
            start = pos + length - k
            if 2 * k > best_len:                 # dấu '>' giữ xâu xuất hiện sớm nhất khi bằng nhau
                best_len, best_text = 2 * k, s[start:start + 2 * k]
        pos += length
    return best_text

assert longest_balanced_text("01000111") == "000111"
assert longest_balanced_text("00111") == "0011"
assert longest_balanced_text("111") == ""
assert longest_balanced_text("00110011") == "0011"
# đối chiếu với độ dài từ lời giải chính trên mọi xâu nhỏ
for length in range(1, 11):
    for bits in product("01", repeat=length):
        t = "".join(bits)
        text = longest_balanced_text(t)
        assert len(text) == Solution().findTheLongestBalancedSubstring(t)
        k = len(text) // 2
        assert text == "0" * k + "1" * k and (text == "" or text in t)
print("tất cả đáp án bài tập: PASS")

tất cả đáp án bài tập: PASS


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "hai bộ đếm một lượt":**

```python
zeros = ones = best = 0
for ch in s:
    if ch == "0":
        if ones > 0:            # khối 0 mới sau khi đã thấy 1
            zeros = ones = 0
        zeros += 1
    else:
        ones += 1
        best = max(best, 2 * min(zeros, ones))
return best
```

**Template "thử mẫu từ lớn xuống":**

```python
for k in range(len(s) // 2, 0, -1):
    if "0" * k + "1" * k in s:
        return 2 * k
return 0
```

**Checklist trước khi submit:**

- [ ] Hiểu xâu cân bằng là `0^k 1^k` (mọi `0` trước mọi `1`, số lượng bằng nhau)?
- [ ] Reset **cả hai** bộ đếm khi gặp `0` mà trước đó đã có `1`?
- [ ] Trả `2 * min(zeros, ones)` (nhân đôi), không phải `min`?
- [ ] `best` khởi tạo `0` (xâu rỗng cân bằng; `"111"` trả `0`)?
- [ ] Đã thử `"10"`, `"0101"`, `"010011"`, `"0"`, `"1"`, độ dài 50?
- [ ] Đã vét cạn mọi xâu nhỏ để đối chiếu?
- [ ] Chữ ký đúng `findTheLongestBalancedSubstring(self, s: str) -> int`?